# CS672 Project #3: Flower Classification with Transfer Learning

**Course:** CS672 - Introduction to Deep Learning  
**Professor:** Sarbanes  
**Due Date:** December 15, 2025  

This notebook builds a transfer learning model for flower classification using both TensorFlow and PyTorch.

**Dataset:** Flowers Recognition (5 classes: daisy, dandelion, rose, sunflower, tulip)  
**Models:** ResNet50 (both TensorFlow and PyTorch)  
**Metrics:** Accuracy, Precision, Recall, F1-Score


## Step 0: Environment Setup & Dataset Upload

Run this cell first to upload your `archive.zip` file containing the flowers dataset.

In [ ]:
# Upload dataset
from google.colab import files
import zipfile
import os

print("Please upload archive.zip containing the flowers dataset...")
uploaded = files.upload()

# Get the uploaded zip filename
zip_name = list(uploaded.keys())[0]
print(f"\nUnzipping {zip_name} to /content...")

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')

print("✓ Extraction complete")

In [ ]:
# Auto-detect flowers folder
import os

# Find the flowers directory
candidates = []
for root, dirs, files_in in os.walk('/content'):
    if set(['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']).issubset(set(dirs)):
        candidates.append(root)
        
if len(candidates) == 0:
    print("⚠️ Could not find flowers folder. Using default path...")
    BASE_DIR = '/content/flowers'
else:
    BASE_DIR = candidates[0]

print(f"\nDataset location: {BASE_DIR}")

# Verify the structure
FLOWER_DIRS = {
    'Daisy': os.path.join(BASE_DIR, 'daisy'),
    'Dandelion': os.path.join(BASE_DIR, 'dandelion'),
    'Rose': os.path.join(BASE_DIR, 'rose'),
    'Sunflower': os.path.join(BASE_DIR, 'sunflower'),
    'Tulip': os.path.join(BASE_DIR, 'tulip'),
}

print("\nVerifying class directories:")
for name, path in FLOWER_DIRS.items():
    exists = os.path.exists(path)
    status = '✓' if exists else '✗'
    print(f"  {status} {name:12s} -> {exists}")

## Step 1: Data Preparation

Load images, resize to 150×150, normalize to [0,1], and prepare labels.

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

IMG_SIZE = 150
X = []  # Images
Z = []  # Labels

def assign_label(img, flower_type):
    """Assign flower type as label"""
    return flower_type

def make_train_data(flower_type, DIR):
    """Load and preprocess images from a directory"""
    if not os.path.exists(DIR):
        print(f"⚠️  {flower_type} directory not found. Skipping...")
        return
    
    files_list = os.listdir(DIR)
    print(f"Processing {flower_type}: {len(files_list)} images")
    
    for img_name in tqdm(files_list):
        try:
            path = os.path.join(DIR, img_name)
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            
            if img is None:
                continue
            
            # Resize to 150x150
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(np.array(img))
            Z.append(flower_type)
        except Exception as e:
            continue

# Load all flower classes
print("\n" + "="*80)
print("STEP 1: DATA PREPARATION")
print("="*80)
print("\nLoading images...\n")

for flower_type, flower_dir in FLOWER_DIRS.items():
    make_train_data(flower_type, flower_dir)

if len(X) == 0:
    raise RuntimeError("No images loaded! Check your dataset path.")

# Convert to numpy array
X = np.array(X, dtype=np.float32)
print(f"\n✓ Total images loaded: {len(X)}")
print(f"✓ Data shape: {X.shape}")

## Step 2: Label Encoding & Train-Test Split

Encode labels, normalize images to [0,1], and split into 75% train / 25% test.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

print("\n" + "="*80)
print("STEP 2: LABEL ENCODING & TRAIN-TEST SPLIT")
print("="*80)

# Normalize images to [0, 1]
X = X / 255.0
print(f"\n✓ Images normalized to [0, 1]")
print(f"  Min value: {X.min():.4f}, Max value: {X.max():.4f}")

# Label encode
le = LabelEncoder()
Y = le.fit_transform(Z)

print(f"\nClasses: {list(le.classes_)}")
print(f"Mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

# Split into train/test (75/25)
x_train, x_test, y_train_labels, y_test_labels = train_test_split(
    X, Y, test_size=0.25, random_state=42, stratify=Y
)

print(f"\n✓ Train-Test Split (75/25):")
print(f"  x_train shape: {x_train.shape}")
print(f"  x_test shape: {x_test.shape}")
print(f"  y_train_labels shape: {y_train_labels.shape}")
print(f"  y_test_labels shape: {y_test_labels.shape}")

# Convert labels to one-hot for TensorFlow
y_train_tf = to_categorical(y_train_labels, 5)
y_test_tf = to_categorical(y_test_labels, 5)

print(f"\n✓ One-hot encoded labels:")
print(f"  y_train_tf shape: {y_train_tf.shape}")
print(f"  y_test_tf shape: {y_test_tf.shape}")

## Step 3: TensorFlow Transfer Learning (ResNet50)

Load pre-trained ResNet50, freeze base layers, add custom head, and train.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("\n" + "="*80)
print("STEP 3: TENSORFLOW RESNET50 TRANSFER LEARNING")
print("="*80)

print("\n✓ Loading pre-trained ResNet50 from ImageNet...")

# Load ResNet50 without top layers
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze all base model layers
for layer in base_model.layers:
    layer.trainable = False

print(f"✓ Base model frozen. Total layers: {len(base_model.layers)}")

In [ ]:
# Build custom classification head
print("\nBuilding custom classification head...")

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(5, activation='softmax')(x)

# Create complete model
model_tf = Model(inputs=base_model.input, outputs=predictions)

# Compile
model_tf.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model created and compiled")

In [ ]:
# Print model summary (REQUIRED by assignment)
print("\n" + "="*80)
print("TENSORFLOW RESNET50 MODEL SUMMARY")
print("="*80 + "\n")

model_tf.summary()

In [ ]:
# Define callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

# Train the model
print("\nTraining TensorFlow ResNet50 model (20 epochs)...\n")

history_tf = model_tf.fit(
    x_train, y_train_tf,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n✓ TensorFlow training complete!")

## Step 4: PyTorch Transfer Learning (ResNet50)

Load pre-trained ResNet50, freeze weights, replace classifier, and train.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models

print("\n" + "="*80)
print("STEP 4: PYTORCH RESNET50 TRANSFER LEARNING")
print("="*80)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")

# Convert numpy to PyTorch tensors (reshape from NHWC to NCHW)
print("\nConverting data to PyTorch format...")
x_train_pt = torch.FloatTensor(x_train.transpose(0, 3, 1, 2))
x_test_pt = torch.FloatTensor(x_test.transpose(0, 3, 1, 2))
y_train_pt = torch.LongTensor(y_train_labels)
y_test_pt = torch.LongTensor(y_test_labels)

print(f"✓ x_train_pt shape: {x_train_pt.shape}")
print(f"✓ y_train_pt shape: {y_train_pt.shape}")

# Create DataLoaders
train_dataset = TensorDataset(x_train_pt, y_train_pt)
test_dataset = TensorDataset(x_test_pt, y_test_pt)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"\n✓ DataLoaders created")
print(f"  Training batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Load ResNet50
print("\nLoading pre-trained ResNet50...")
resnet_pt = models.resnet50(pretrained=True)
num_features = resnet_pt.fc.in_features

# Freeze all parameters
for param in resnet_pt.parameters():
    param.requires_grad = False

print(f"✓ ResNet50 loaded and frozen")
print(f"  Input features to classifier: {num_features}")

# Replace fully connected layer
resnet_pt.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 5)
)

# Move to device
resnet_pt = resnet_pt.to(device)

print(f"✓ Custom classifier added")

In [ ]:
# Print model summary (REQUIRED by assignment)
print("\n" + "="*80)
print("PYTORCH RESNET50 MODEL SUMMARY")
print("="*80 + "\n")

print(resnet_pt)

# Count parameters
total_params = sum(p.numel() for p in resnet_pt.parameters())
trainable_params = sum(p.numel() for p in resnet_pt.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nParameter Summary:")
print(f"  Total Parameters: {total_params:,}")
print(f"  Trainable Parameters: {trainable_params:,}")
print(f"  Frozen Parameters: {frozen_params:,}")

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_pt.parameters(), lr=0.001)

# Training loop
num_epochs = 20
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("\nTraining PyTorch ResNet50 model (20 epochs)...\n")

for epoch in range(num_epochs):
    # Training phase
    resnet_pt.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = resnet_pt(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total

    # Validation phase
    resnet_pt.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet_pt(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(test_dataset)
    val_acc = val_correct / val_total

    train_losses.append(epoch_loss)
    val_losses.append(val_loss)
    train_accs.append(epoch_acc)
    val_accs.append(val_acc)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f}, "
              f"Train Acc: {epoch_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print("\n✓ PyTorch training complete!")

## Step 5: Model Evaluation

Evaluate both models and compute metrics: Accuracy, Precision, Recall, F1-Score.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("\n" + "="*80)
print("STEP 5: MODEL EVALUATION")
print("="*80)

In [ ]:
# Evaluate TensorFlow model
print("\nEvaluating TensorFlow ResNet50...\n")

test_loss_tf, test_acc_tf = model_tf.evaluate(x_test, y_test_tf, verbose=0)
print(f"Test Loss: {test_loss_tf:.4f}")
print(f"Test Accuracy: {test_acc_tf:.4f}")

# Make predictions
y_pred_tf_prob = model_tf.predict(x_test, verbose=0)
y_pred_tf = np.argmax(y_pred_tf_prob, axis=1)

# Calculate metrics
accuracy_tf = accuracy_score(y_test_labels, y_pred_tf)
precision_tf = precision_score(y_test_labels, y_pred_tf, average='weighted', zero_division=0)
recall_tf = recall_score(y_test_labels, y_pred_tf, average='weighted', zero_division=0)
f1_tf = f1_score(y_test_labels, y_pred_tf, average='weighted', zero_division=0)

print(f"\nTensorFlow Metrics:")
print(f"  Accuracy : {accuracy_tf:.4f}")
print(f"  Precision: {precision_tf:.4f}")
print(f"  Recall   : {recall_tf:.4f}")
print(f"  F1-Score : {f1_tf:.4f}")

In [ ]:
# Print detailed classification report
print("\nTensorFlow Classification Report:")
print("="*50)
print(classification_report(y_test_labels, y_pred_tf, target_names=le.classes_, zero_division=0))

In [ ]:
# Evaluate PyTorch model
print("\nEvaluating PyTorch ResNet50...\n")

resnet_pt.eval()
y_pred_pt = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = resnet_pt(images)
        _, predicted = torch.max(outputs, 1)
        y_pred_pt.extend(predicted.cpu().numpy())

y_pred_pt = np.array(y_pred_pt)

# Calculate metrics
accuracy_pt = accuracy_score(y_test_labels, y_pred_pt)
precision_pt = precision_score(y_test_labels, y_pred_pt, average='weighted', zero_division=0)
recall_pt = recall_score(y_test_labels, y_pred_pt, average='weighted', zero_division=0)
f1_pt = f1_score(y_test_labels, y_pred_pt, average='weighted', zero_division=0)

print(f"PyTorch Metrics:")
print(f"  Accuracy : {accuracy_pt:.4f}")
print(f"  Precision: {precision_pt:.4f}")
print(f"  Recall   : {recall_pt:.4f}")
print(f"  F1-Score : {f1_pt:.4f}")

In [ ]:
# Print detailed classification report
print("\nPyTorch Classification Report:")
print("="*50)
print(classification_report(y_test_labels, y_pred_pt, target_names=le.classes_, zero_division=0))

## Step 6: Visualization

Create confusion matrices and training history plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*80)
print("STEP 6: VISUALIZATION")
print("="*80)

# TensorFlow Confusion Matrix
cm_tf = confusion_matrix(y_test_labels, y_pred_tf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_tf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('TensorFlow ResNet50 - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('tf_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: tf_confusion_matrix.png")

In [ ]:
# PyTorch Confusion Matrix
cm_pt = confusion_matrix(y_test_labels, y_pred_pt)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_pt, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('PyTorch ResNet50 - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('pt_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: pt_confusion_matrix.png")

In [ ]:
# TensorFlow Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(history_tf.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_tf.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('TensorFlow ResNet50 - Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history_tf.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history_tf.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('TensorFlow ResNet50 - Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tf_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: tf_training_history.png")

In [ ]:
# PyTorch Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('PyTorch ResNet50 - Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label='Training Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].set_title('PyTorch ResNet50 - Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pt_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: pt_training_history.png")

## Step 7: Save Models

Save both models for deployment.

In [ ]:
print("\n" + "="*80)
print("STEP 7: MODEL SAVING")
print("="*80)

# Save TensorFlow model
model_tf.save('flower_classifier_tf_resnet50.h5')
print("\n✓ Saved: flower_classifier_tf_resnet50.h5")

# Save PyTorch model
torch.save(resnet_pt.state_dict(), 'flower_classifier_pt_resnet50.pth')
print("✓ Saved: flower_classifier_pt_resnet50.pth")

## Summary

Project complete! Here's what was accomplished:

In [ ]:
import pandas as pd

print("\n" + "="*80)
print("PROJECT SUMMARY")
print("="*80)

summary_data = {
    'Metric': ['Test Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'TensorFlow (ResNet50)': [f'{accuracy_tf:.4f}', f'{precision_tf:.4f}', 
                               f'{recall_tf:.4f}', f'{f1_tf:.4f}'],
    'PyTorch (ResNet50)': [f'{accuracy_pt:.4f}', f'{precision_pt:.4f}', 
                          f'{recall_pt:.4f}', f'{f1_pt:.4f}']
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("FILES GENERATED:")
print("="*80)
print("\nModels:")
print("  • flower_classifier_tf_resnet50.h5")
print("  • flower_classifier_pt_resnet50.pth")
print("\nVisualizations:")
print("  • tf_confusion_matrix.png")
print("  • pt_confusion_matrix.png")
print("  • tf_training_history.png")
print("  • pt_training_history.png")
print("\n✓ Project complete! Due: December 15, 2025")